# 날씨 기반 판매수량 예측 & 발주 추천 — Colab A100

**TFT · Quantile Loss · Walmart Stormy Weather · 한국어 실행 가이드**

매장×상품별 지난 **56일**의 판매·날씨로 앞으로 **7일**의 판매수량을 예측합니다.
출력은 Q10/Q25/Q50/Q75/Q90이며, 재고·입고 예정량·납기·발주 주기를 입력해 발주 후보를 계산합니다.

> 이 파일은 실행 가능한 연구용 출발점입니다. Walmart 모델을 국내 소상공인에 바로 적용해 매출 증가를 보장하지 않습니다.
> **판매량은 수요와 다릅니다.** 품절 시 관측 판매량은 실제 수요보다 작습니다. 실매장 적용에는 POS 판매수량,
> 품절·재고·가격·행사·휴무·지역 날씨와 실제 예보 이력을 모아 재학습하고 발주 성과를 검증해야 합니다.

### 먼저 할 일
1. Colab에서 이 `.ipynb`를 업로드합니다. **런타임 → 런타임 유형 변경 → Python 3 → GPU → A100**을 선택합니다.
   가능하면 고용량 RAM도 선택합니다. Colab Pro여도 A100 할당·사용 시간은 보장되지 않습니다.
2. 아래 **설치 셀만 실행**하고, **런타임 → 세션 다시 시작** 후 **설정 셀부터** 차례로 실행합니다.
3. [Kaggle 대회](https://www.kaggle.com/competitions/walmart-recruiting-sales-in-stormy-weather/data)에 로그인하고
   필요한 대회 이용 조건에 동의한 뒤 파일을 받습니다. 기본 방식은 **ZIP 또는 `train.csv`, `weather.csv`, `key.csv` 직접 업로드**입니다.
   `test.csv`에는 정답이 없으므로 여기서는 `train.csv`의 마지막 기간으로 검증·테스트합니다.
4. GPU 비용을 쓰기 전에 `SMOKE_TEST=True`로 실행 흐름을 확인할 수 있습니다. 이 모드의 데이터·성능은 **합성 테스트**입니다.
5. 실제 실험은 `SMOKE_TEST=False`. 먼저 `MAX_SERIES=50`, `MAX_EPOCHS=5`로 확인하고 200개·20에포크로 늘리세요.

### 데이터 선택과 의미
| 파일 | 필수 열 | 역할 |
|---|---|---|
| `train.csv` | `date, store_nbr, item_nbr, units` | 일별 판매 개수(학습 목표) |
| `key.csv` | `store_nbr, station_nbr` | 매장과 기상 관측소 연결 |
| `weather.csv` | `date, station_nbr, tavg, tmax, tmin, preciptotal, avgspeed` | 날씨 설명 변수 |

다른 후보 데이터셋은 열 구조·시간 간격·단위가 달라 그대로 바꿔 넣을 수 없습니다.
특히 주별 매출액 `Weekly_Sales`와 일별 판매수량 `units`는 서로 다른 목표입니다.
기온은 원자료의 °F, 강수는 inch, 풍속은 mph를 유지합니다. 국내 데이터는 같은 단위로 맞추거나 전체 전처리를 바꿔 재학습하세요.

### 분위수 손실의 정확한 의미
잔차 $e=y-\hat y_q$에 대해 $L_q(e)=\max(qe,(q-1)e)$입니다.
MSE 최적 예측은 조건부 평균, MAE는 조건부 중앙값이며, Quantile Loss는 지정한 **조건부 분위수**를 학습합니다.
여기서는 $0<q<1$인 5개 분위수를 **학습 전에 고정**합니다. 임의 분위수를 나중에 정확히 얻는 기능은 아닙니다.
Q90이 적절히 보정되었다면 관측값 약 90%가 그 이하가 되며, 개별 날짜의 성공 확률을 보장하지 않습니다.
PyTorch Forecasting의 QuantileLoss는 통상 pinball loss의 2배 관례를 사용하므로 아래 직접 계산한 pinball과 로그 값의 척도가 다를 수 있습니다.


In [ ]:
# 1. 새 Colab GPU 런타임에서 한 번만 실행 → 세션 다시 시작
# Colab이 제공하는 CUDA용 torch는 유지합니다. CPU용 torch로 교체하지 않습니다.
import subprocess, sys
packages = [
    "pytorch-forecasting==1.5.0", "lightning==2.5.5",
    "pytorch-lightning==2.5.5", "torchmetrics==1.7.4",
    "pandas>=2.2,<3", "numpy>=1.26,<3", "scikit-learn>=1.4,<2",
    "matplotlib>=3.8,<4", "kaggle>=1.8,<2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("설치 완료. 런타임 → 세션 다시 시작 후 다음 설정 셀부터 실행하세요.")


In [ ]:
# 2. 사용자 설정 — 주로 이 셀만 수정합니다.
SMOKE_TEST = False           # True: 외부 데이터 없이 합성 데이터로 1에포크 검사
DATA_SOURCE = "upload"      # "upload" / "kaggle" / "drive"
MOUNT_DRIVE = True           # Colab 세션 종료 뒤에도 체크포인트와 결과 보존
DRIVE_PROJECT = "/content/drive/MyDrive/weather_order_tft"
DRIVE_DATA_SUBDIR = "data"   # drive 방식: 이 하위 폴더에 CSV 또는 ZIP 저장
REQUIRE_A100 = True          # 다른 GPU를 허용하려면 False. smoke 모드는 CPU도 허용

SEED = 42
ENCODER_DAYS = 56
HORIZON = 7
VALID_DAYS = 56
TEST_DAYS = 56
MAX_SERIES = 200             # None: 학습 조건을 만족하는 모든 매장×상품
MIN_NONZERO_DAYS = 14        # 학습 구간 기준. 간헐 수요 전체 평가 시 낮추세요.
BATCH_SIZE = 128             # GPU 메모리 부족 시 64 → 32
MAX_EPOCHS = 20
HIDDEN_SIZE = 64
NUM_WORKERS = 2              # Colab worker 오류면 0
TRAIN_BATCH_LIMIT = 1.0      # float 1.0: 전체; int 200: 에포크당 200배치
RUN_NO_WEATHER_ABLATION = False  # True: 날씨 없는 TFT도 학습·테스트해 기여도 비교
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90]

# 데이터 마지막 관측일 다음 날부터 7일 예측. 실제 '오늘' 예측과 다릅니다.
# 실매장 운영 시 아래 CSV에 발행 시점까지 검증 가능한 미래 예보를 제공하세요.
FUTURE_WEATHER_CSV = ""       # 비워 두면 마지막 관측 날씨 유지 가정
INVENTORY_CSV = ""            # 비워 두면 명시적으로 표시된 예시 재고만 사용

# CPU smoke용 축소 설정
if SMOKE_TEST:
    MOUNT_DRIVE = False
    MAX_SERIES, MIN_NONZERO_DAYS = 8, 3
    ENCODER_DAYS, VALID_DAYS, TEST_DAYS = 28, 28, 28
    BATCH_SIZE, HIDDEN_SIZE, MAX_EPOCHS, NUM_WORKERS = 16, 8, 1, 0
    TRAIN_BATCH_LIMIT = 3
    RUN_NO_WEATHER_ABLATION = False


In [ ]:
# 3. import / GPU / 결과 폴더
import os, sys, json, math, zipfile, shutil, pickle, warnings, gc, platform
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data.encoders import GroupNormalizer, NaNLabelEncoder
from pytorch_forecasting.metrics import QuantileLoss
from IPython.display import display

pl.seed_everything(SEED, workers=True)
assert 1 <= HORIZON <= ENCODER_DAYS
assert 0.5 in QUANTILES and all(0 < q < 1 for q in QUANTILES)
assert QUANTILES == sorted(set(QUANTILES))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
if not SMOKE_TEST:
    assert torch.cuda.is_available(), "GPU 런타임으로 바꿔 주세요."
    assert not REQUIRE_A100 or "A100" in gpu, f"현재 {gpu}. A100 선택 또는 REQUIRE_A100=False"
PRECISION = "bf16-mixed" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "32-true"
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
if SMOKE_TEST and not torch.cuda.is_available():
    torch.set_num_threads(2)
print(f"Python {platform.python_version()} | torch {torch.__version__} | {gpu} | {PRECISION}")
print({p: version(p) for p in ["pytorch-forecasting", "lightning", "torchmetrics", "pandas"]})

IN_COLAB = "google.colab" in sys.modules
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(DRIVE_PROJECT if MOUNT_DRIVE else ("/content/weather_order_tft" if IN_COLAB else "weather_order_tft"))
DATA_DIR = Path("/content/weather_order_data" if IN_COLAB else "weather_order_data")
RUN_NAME = ("smoke_" if SMOKE_TEST else "walmart_") + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = ROOT / "runs" / RUN_NAME
for p in [ROOT, DATA_DIR, OUT]:
    p.mkdir(parents=True, exist_ok=True)
print("결과 저장:", OUT)


## 데이터 가져오기
기본 `upload`에서는 ZIP 하나 또는 세 CSV를 선택합니다. 중첩 ZIP도 해제합니다.
`drive` 방식은 `MyDrive/weather_order_tft/data` 안의 파일을 사용합니다.

`kaggle` 방식은 먼저 Colab 왼쪽 **열쇠 아이콘 → Secrets**에 `KAGGLE_API_TOKEN`을 넣고 이 노트북의 접근을 허용하세요.
구형 인증은 `KAGGLE_USERNAME`, `KAGGLE_KEY` 두 Secret도 지원합니다. 키를 코드에 쓰거나 출력하지 않습니다.
403 오류는 대회 규칙 동의 또는 계정 권한을 확인하고, 내려받기가 불가능하면 직접 업로드하세요.


In [ ]:
# 4. 업로드 / Kaggle / Drive / 합성 테스트 데이터
def extract_zips(root):
    handled = set()
    while True:
        archives = [p for p in root.rglob("*.zip") if p.resolve() not in handled]
        if not archives:
            break
        for archive in archives:
            handled.add(archive.resolve())
            dest = archive.parent / (archive.stem + "_unpacked")
            dest.mkdir(exist_ok=True)
            with zipfile.ZipFile(archive) as z:
                for info in z.infolist():
                    target = (dest / info.filename).resolve()
                    if not target.is_relative_to(dest.resolve()):
                        raise ValueError("ZIP 경로가 추출 폴더를 벗어납니다.")
                z.extractall(dest)

def find_csv(root, name):
    matches = [p for p in root.rglob("*") if p.is_file() and p.name.lower() == name]
    if len(matches) != 1:
        raise ValueError(f"{name}: 정확히 1개가 필요합니다. 발견 {len(matches)}개. 데이터 폴더의 중복/누락을 확인하세요.")
    return matches[0]

def synthetic_data():
    rng = np.random.default_rng(SEED)
    dates = pd.date_range("2022-01-01", periods=400)
    key = pd.DataFrame({"store_nbr": [1, 2], "station_nbr": [1, 2]})
    ws, ss = [], []
    for station in [1, 2]:
        t = 60 + 20*np.sin(np.arange(len(dates))*2*np.pi/365) + rng.normal(0, 4, len(dates))
        rain = rng.gamma(1, .2, len(dates)) * (rng.random(len(dates)) < .25)
        ws.append(pd.DataFrame({"date": dates, "station_nbr": station, "tavg": t,
            "tmax": t+8, "tmin": t-8, "preciptotal": rain,
            "avgspeed": rng.uniform(1, 12, len(dates))}))
        for item in range(1, 5):
            mu = np.maximum(1, 8+item*4 + .3*(t-60) + 6*(dates.dayofweek>=5) - rain*5)
            units = rng.poisson(mu)
            s = pd.DataFrame({"date": dates, "store_nbr": station, "item_nbr": item, "units": units})
            # 실제 0과 기록 누락을 구별하는지 검사할 구간
            s.loc[::47, "units"] = 0
            ss.append(s.drop(index=[150, 151, 330]))
    weather = pd.concat(ws, ignore_index=True).astype({"preciptotal": str, "tavg": str})
    weather.loc[10, "preciptotal"] = "T"
    weather.loc[20, "tavg"] = "M"
    return pd.concat(ss, ignore_index=True), weather, key

if SMOKE_TEST:
    sales_raw, weather_raw, key_raw = synthetic_data()
    print("합성 데이터 — 실제 성능으로 인용하지 마세요.")
else:
    if DATA_SOURCE == "drive":
        assert MOUNT_DRIVE, "drive 방식은 MOUNT_DRIVE=True 필요"
        DATA_DIR = ROOT / DRIVE_DATA_SUBDIR
        assert DATA_DIR.exists(), f"폴더에 데이터를 넣으세요: {DATA_DIR}"
    elif DATA_SOURCE == "upload":
        existing = [p for p in DATA_DIR.rglob("train.csv")]
        if not existing:
            from google.colab import files
            uploaded = files.upload()
            for name, content in uploaded.items():
                (DATA_DIR / Path(name).name).write_bytes(content)
            del uploaded
    elif DATA_SOURCE == "kaggle":
        from google.colab import userdata
        env = os.environ.copy()
        for name in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
            try:
                env[name] = userdata.get(name)
            except Exception:
                pass
        assert env.get("KAGGLE_API_TOKEN") or (env.get("KAGGLE_USERNAME") and env.get("KAGGLE_KEY")), "Kaggle Secret를 설정하세요."
        import subprocess
        subprocess.run(["kaggle", "competitions", "download", "-c",
            "walmart-recruiting-sales-in-stormy-weather", "-p", str(DATA_DIR)], check=True, env=env)
        del env
    else:
        raise ValueError("DATA_SOURCE는 upload / kaggle / drive 중 선택")
    extract_zips(DATA_DIR)
    sales_raw = pd.read_csv(find_csv(DATA_DIR, "train.csv"))
    weather_raw = pd.read_csv(find_csv(DATA_DIR, "weather.csv"), low_memory=False)
    key_raw = pd.read_csv(find_csv(DATA_DIR, "key.csv"))
print("원본 행 수:", {"sales": len(sales_raw), "weather": len(weather_raw), "key": len(key_raw)})


## 누수 방지와 결측 처리
- 시간 순서대로 **학습 → 검증 56일 → 테스트 56일**을 나눕니다. 검증은 체크포인트 선택·기준선 보정용, 테스트는 최종 평가용입니다.
- 매장×상품 선택, 날씨 대체 중앙값, 정규화는 학습 구간만 사용합니다.
- 판매 기록이 없는 날은 판매 0으로 간주하지 않습니다. 과거 입력만 이전 값으로 채우고 `sales_observed`를 함께 넣습니다.
  **정답 7일에 누락이 하나라도 있는 창은 학습·평가에서 제외**합니다. 실제 0판매는 유지합니다.
- 날씨 `M`은 결측, 강수 `T`는 미량으로 보고 수치 0과 별도 `precip_trace`로 표현합니다.
  미래 값으로 역방향 채우기(`bfill`)하지 않습니다. 원본 관측 날씨는 날짜·관측소별로 결합합니다.
- 각 배치의 미래 날씨를 **예측 시작 직전 마지막 관측값으로 강제 교체**합니다.
  따라서 미래 실제 날씨를 미리 아는 평가가 아닙니다. 이는 지속성 예보 대용치이지 실제 기상 예보가 아닙니다.
- 하루 영업 종료 후 당일 판매·날씨가 확보된 시점을 예측 시점으로 가정합니다. 실제 관측 지연이 있으면 추가로 시차를 주어야 합니다.


In [ ]:
# 5. 스키마 검사와 시간 분할 — 랜덤 분할 금지
WEATHER_BASE = ["tavg", "tmax", "tmin", "preciptotal", "avgspeed"]
GROUP = ["store_nbr", "item_nbr"]
def require_columns(df, names, label):
    missing = set(names) - set(df.columns)
    if missing:
        raise ValueError(f"{label} 필수 열 누락: {sorted(missing)}")
require_columns(sales_raw, ["date", *GROUP, "units"], "train.csv")
require_columns(weather_raw, ["date", "station_nbr", *WEATHER_BASE], "weather.csv")
require_columns(key_raw, ["store_nbr", "station_nbr"], "key.csv")
sales = sales_raw[["date", *GROUP, "units"]].copy()
weather = weather_raw[["date", "station_nbr", *WEATHER_BASE]].copy()
key = key_raw[["store_nbr", "station_nbr"]].copy()
for frame in [sales, weather]:
    frame["date"] = pd.to_datetime(frame["date"], errors="raise").dt.normalize()
for frame, cols in [(sales, GROUP), (weather, ["station_nbr"]), (key, ["store_nbr", "station_nbr"])]:
    for col in cols:
        assert frame[col].notna().all(), f"{col} 식별자 결측"
        frame[col] = frame[col].astype(str)
assert not sales.duplicated([*GROUP, "date"]).any(), "판매 중복 키: 합산/중복 여부를 직접 검토하세요."
assert not weather.duplicated(["station_nbr", "date"]).any(), "날씨 중복 키"
assert not key.duplicated("store_nbr").any(), "매장별 관측소는 하나여야 합니다."
sales["units"] = pd.to_numeric(sales["units"], errors="raise")
assert np.isfinite(sales["units"]).all() and (sales["units"] >= 0).all(), "판매수량 결측/음수/무한대 확인"
assert np.allclose(sales["units"], sales["units"].round()), "금액 대신 판매 개수 units가 필요합니다."
START_DATE, END_DATE = sales.date.min(), sales.date.max()
TEST_START = END_DATE - pd.Timedelta(days=TEST_DAYS-1)
VALID_START = TEST_START - pd.Timedelta(days=VALID_DAYS)
TRAIN_END = VALID_START - pd.Timedelta(days=1)
VALID_END = TEST_START - pd.Timedelta(days=1)
assert (TRAIN_END - START_DATE).days >= ENCODER_DAYS + HORIZON + 30, "학습 기간이 너무 짧습니다."
train_raw = sales[sales.date <= TRAIN_END]
stats = train_raw.groupby(GROUP).units.agg(total="sum", count="size", positive=lambda x: (x>0).sum())
eligible = stats[(stats.positive >= MIN_NONZERO_DAYS) & (stats['count'] >= ENCODER_DAYS + HORIZON)].sort_values("total", ascending=False)
if MAX_SERIES is not None:
    eligible = eligible.head(MAX_SERIES)
assert len(eligible), "학습 가능한 시계열이 없습니다. 필터를 확인하세요."
selected = eligible.reset_index()[GROUP]
sales = sales.merge(selected, on=GROUP, validate="many_to_one")
series_map = selected.merge(key, on="store_nbr", validate="many_to_one")
assert len(series_map) == len(selected), "관측소 연결 누락"
series_map.to_csv(OUT / "selected_series.csv", index=False)
display(pd.DataFrame({"split": ["train", "validation", "test"],
    "start": [START_DATE, VALID_START, TEST_START], "end": [TRAIN_END, VALID_END, END_DATE]}))
print(f"학습 조건에 따라 선택한 매장×상품: {len(selected)} / {len(stats)}")
print("선택된 상품군 기준 성능입니다. 저판매·간헐수요 전체의 성능을 대표하지 않습니다.")


In [ ]:
# 6. 관측소별 날씨 정리 및 일별 판매 패널
stations = series_map.station_nbr.unique()
weather = weather[weather.station_nbr.isin(stations)].copy()
weather["precip_trace"] = weather.preciptotal.astype(str).str.strip().eq("T").astype(float)
for col in WEATHER_BASE:
    weather[col] = pd.to_numeric(weather[col].astype(str).str.strip().replace({"M": np.nan, "T": "0"}), errors="coerce")
weather[WEATHER_BASE] = weather[WEATHER_BASE].replace([np.inf, -np.inf], np.nan)
dates = pd.date_range(START_DATE, END_DATE)
wgrid = pd.MultiIndex.from_product([stations, dates], names=["station_nbr", "date"]).to_frame(index=False)
weather = wgrid.merge(weather, on=["station_nbr", "date"], how="left", validate="one_to_one").sort_values(["station_nbr", "date"])
WEATHER_MISSING = [c + "_missing" for c in WEATHER_BASE]
for col in WEATHER_BASE:
    weather[col + "_missing"] = weather[col].isna().astype(float)
weather["precip_trace"] = weather.precip_trace.fillna(0.0)
weather_defaults = weather.loc[weather.date <= TRAIN_END, WEATHER_BASE].median().fillna(0.0)
weather[WEATHER_BASE] = weather.groupby("station_nbr", observed=True)[WEATHER_BASE].ffill()
weather[WEATHER_BASE] = weather[WEATHER_BASE].fillna(weather_defaults)
WEATHER_FEATURES = [*WEATHER_BASE, *WEATHER_MISSING, "precip_trace"]

grid = selected.merge(pd.DataFrame({"date": dates}), how="cross")
panel = grid.merge(sales, on=[*GROUP, "date"], how="left", validate="one_to_one")
panel = panel.merge(key, on="store_nbr", how="left", validate="many_to_one")
assert panel.station_nbr.notna().all()
panel = panel.merge(weather, on=["station_nbr", "date"], how="left", validate="many_to_one")
panel = panel.sort_values([*GROUP, "date"]).reset_index(drop=True)
panel["sales_observed"] = panel.units.notna().astype(float)
panel["actual_units"] = panel.units  # 평가용 원본 정답. 기록 누락은 NaN으로 유지
panel["units"] = panel.groupby(GROUP, observed=True).units.ffill().fillna(0).astype("float32")
# 최초 관측 이전만 0 자리표시자. sales_observed=0으로 구분하며 정답으로 학습하지 않음
def calendar_features(df):
    out = df.copy()
    out["time_idx"] = (out.date - START_DATE).dt.days.astype("int64")
    out["dow"] = out.date.dt.dayofweek.astype(str)
    out["month"] = out.date.dt.month.astype(str)
    out["weekend"] = (out.date.dt.dayofweek >= 5).astype(float)
    out["year_sin"] = np.sin(2*np.pi*out.date.dt.dayofyear/365.25)
    out["year_cos"] = np.cos(2*np.pi*out.date.dt.dayofyear/365.25)
    return out
panel = calendar_features(panel)
for col in ["units", "sales_observed", "weekend", "year_sin", "year_cos", *WEATHER_FEATURES]:
    panel[col] = panel[col].astype("float32")
assert panel[["units", *WEATHER_FEATURES]].notna().sum().min() == len(panel)
assert panel.groupby(GROUP).time_idx.diff().dropna().eq(1).all()
TRAIN_CUTOFF = (TRAIN_END - START_DATE).days
VAL_CUTOFF = (VALID_END - START_DATE).days
MAX_TIME = (END_DATE - START_DATE).days
print("판매 기록 누락 비율:", round(1-panel.sales_observed.mean(), 4))
print("실제 관측된 0판매 비율:", round((panel.actual_units.dropna()==0).mean(), 4))
display(panel.head(3))


## TFT 데이터와 날씨 입력
모든 창은 과거 56일/미래 7일로 고정합니다. 검증·테스트는 **매주 한 번 예측하는 시점**을 사용하여
같은 날짜의 정답을 여러 예측 시점에서 중복 집계하지 않습니다. 매주 시작점의 7일 정답이 모두 관측된 창만 평가합니다.
테스트 기간 중 다음 주 예측에는 그때까지 관측된 판매 이력을 사용할 수 있습니다. 모델 파라미터는 테스트로 학습하지 않습니다.

날씨를 decoder에 넣는 동작은 `WeatherCollator`가 담당합니다. 모델을 로드한 뒤에도 반드시 `make_loader`를 사용하세요.
직접 `model.predict(dataframe)`를 부르면 이 가공을 건너뛸 수 있습니다.


In [ ]:
# 7. 데이터셋 / 정답 누락 창 제거 / 과거 관측만 사용하는 날씨 collator
CALENDAR_REAL = ["time_idx", "weekend", "year_sin", "year_cos"]
observation_prefix = {}
for group, part in panel.groupby(GROUP, sort=False):
    observation_prefix[tuple(group)] = np.r_[0, part.sales_observed.to_numpy().cumsum()]

def valid_windows(dataset, weekly_start=None):
    def keep(idx):
        flags = []
        for row in idx.itertuples(index=False):
            a, b = int(row.time_idx_first_prediction), int(row.time_idx_last)
            prefix = observation_prefix[(str(row.store_nbr), str(row.item_nbr))]
            good = (b-a+1 == HORIZON) and (prefix[b+1]-prefix[a] == HORIZON)
            if weekly_start is not None:
                good = good and ((a-weekly_start) % HORIZON == 0)
            flags.append(good)
        return np.array(flags, dtype=bool)
    before = len(dataset)
    ds = dataset.filter(keep)
    assert len(ds)>0, "정답이 온전한 창이 없습니다. 기간·HORIZON·자료 누락을 확인하세요."
    print(f"창 유지 {len(ds):,}/{before:,}" + (" (주 단위 평가)" if weekly_start is not None else ""))
    return ds

def build_datasets(with_weather=True):
    weather_cols = WEATHER_FEATURES if with_weather else []
    training = TimeSeriesDataSet(
        panel[panel.time_idx <= TRAIN_CUTOFF].copy(),
        time_idx="time_idx", target="units", group_ids=GROUP,
        min_encoder_length=ENCODER_DAYS, max_encoder_length=ENCODER_DAYS,
        min_prediction_length=HORIZON, max_prediction_length=HORIZON,
        static_categoricals=["store_nbr", "item_nbr", "station_nbr"],
        time_varying_known_categoricals=["dow", "month"],
        time_varying_known_reals=CALENDAR_REAL + weather_cols,
        time_varying_unknown_reals=["units", "sales_observed"],
        target_normalizer=GroupNormalizer(groups=GROUP, transformation=None),
        categorical_encoders={name: NaNLabelEncoder(add_nan=True) for name in [*GROUP, "station_nbr", "dow", "month"]},
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False, randomize_length=False,
    )
    validation = TimeSeriesDataSet.from_dataset(training,
        panel[panel.time_idx <= VAL_CUTOFF].copy(),
        min_prediction_idx=TRAIN_CUTOFF+1, stop_randomization=True, predict=False)
    test = TimeSeriesDataSet.from_dataset(training, panel.copy(),
        min_prediction_idx=VAL_CUTOFF+1, stop_randomization=True, predict=False)
    return (valid_windows(training), valid_windows(validation, TRAIN_CUTOFF+1),
            valid_windows(test, VAL_CUTOFF+1))

class WeatherCollator:
    def __init__(self, reals, mode="persistence"):
        assert mode in {"persistence", "provided"}
        self.indices = [reals.index(c) for c in WEATHER_FEATURES if c in reals]
        self.mode = mode
    def __call__(self, samples):
        x, y = TimeSeriesDataSet._collate_fn(samples)
        if self.mode == "persistence":
            row = torch.arange(x["encoder_lengths"].shape[0])
            for j in self.indices:
                last = x["encoder_cont"][row, x["encoder_lengths"]-1, j]
                x["decoder_cont"][:, :, j] = last[:, None]
        return x, y

def make_loader(ds, train=False, weather_mode="persistence"):
    return ds.to_dataloader(train=train, batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS>0),
        collate_fn=WeatherCollator(ds.reals, weather_mode))

training, validation, testing = build_datasets(True)
inspection_loader = make_loader(validation)
x_check, y_check = next(iter(inspection_loader))
del inspection_loader
for c in WEATHER_FEATURES:
    j = training.reals.index(c)
    assert torch.allclose(x_check["decoder_cont"][:,:,j], x_check["encoder_cont"][:,-1,j,None].expand(-1,HORIZON))
print("검사 통과: 미래 날씨 입력 = 예측 직전 관측값, 실제 미래 날씨 노출 없음")


In [ ]:
# 8. A100 BF16 학습 / 검증 손실로 최적 체크포인트 저장
def fit_tft(ds_train, ds_val, label):
    pl.seed_everything(SEED, workers=True)
    ckpt = ModelCheckpoint(dirpath=OUT / label / "checkpoints",
        filename="best-{epoch:02d}-{val_loss:.4f}", monitor="val_loss", mode="min", save_top_k=1, save_last=True)
    model = TemporalFusionTransformer.from_dataset(ds_train,
        learning_rate=1e-3, hidden_size=HIDDEN_SIZE, attention_head_size=4,
        dropout=0.15, hidden_continuous_size=min(32, HIDDEN_SIZE),
        lstm_layers=1, output_size=len(QUANTILES),
        loss=QuantileLoss(quantiles=QUANTILES), optimizer="adam",
        reduce_on_plateau_patience=3, log_interval=-1,
        mask_bias=-float("inf"),
    )
    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1,
        precision=PRECISION, max_epochs=MAX_EPOCHS,
        gradient_clip_val=0.5, limit_train_batches=TRAIN_BATCH_LIMIT,
        limit_val_batches=1.0, num_sanity_val_steps=2,
        callbacks=[ckpt, EarlyStopping(monitor="val_loss", patience=5, mode="min", check_finite=True),
                   LearningRateMonitor(logging_interval="epoch")],
        logger=CSVLogger(str(OUT / label), name="logs"),
        log_every_n_steps=10, enable_model_summary=False,
    )
    print(f"{label}: {model.size()/1e6:.2f}M parameters")
    trainer.fit(model, train_dataloaders=make_loader(ds_train, True), val_dataloaders=make_loader(ds_val))
    assert ckpt.best_model_path, "저장된 체크포인트가 없습니다. 학습 로그를 확인하세요."
    # 이번 실행에서 직접 생성한 체크포인트만 로드
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path, map_location="cpu")
    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best, ckpt.best_model_path

best_model, best_checkpoint = fit_tft(training, validation, "tft_weather")
print("최적 모델:", best_checkpoint)


## 평가: 단순 기준선보다 나은지 확인
기준선은 **예측 시점에 이미 알려진 마지막 7일 판매 패턴을 반복**합니다.
기준선의 분위수는 검증 기간의 잔차를 예측일차별로 보정하고, 테스트에서는 고정합니다.
MAE, RMSE, WAPE, 평균 pinball loss, Q10–Q90 포함률과 폭, 각 분위수 이하 관측 비율을 확인합니다.
판매 0이 많으므로 MAPE는 쓰지 않습니다. WAPE 분모가 0인 경우 NaN으로 표시합니다.

분위수 교차는 발생할 수 있어 원출력을 보존하고, 사용용 출력은 0 하한·분위수 정렬을 적용합니다.
후처리 전후 지표를 모두 저장합니다. **Q10–Q90 포함률의 목표는 80%**지만 자동 보장되지 않습니다.


In [ ]:
# 9. 배치 추론 — return_x를 전체 누적하지 않아 RAM 사용을 줄입니다.
QCOLS = [f"q{int(round(q*100)):02d}" for q in QUANTILES]
QMED = QCOLS[QUANTILES.index(0.5)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def batch_forecasts(model, ds, weather_mode="persistence", with_targets=True):
    model = model.to(device).eval()
    rows = []
    with torch.inference_mode():
        for x, y in make_loader(ds, weather_mode=weather_mode):
            idx = ds.x_to_index(x)
            xx = {k: v.to(device) if torch.is_tensor(v) else v for k, v in x.items()}
            # 추론은 FP32. BF16 학습 후에도 지표 계산의 수치 안정성을 유지
            pred = model(xx)["prediction"].float().cpu().numpy()
            assert np.isfinite(pred).all(), "예측에 NaN/Inf가 있습니다. 학습률·precision을 확인하세요."
            hist = x["encoder_target"].float().cpu().numpy()
            baseline = hist[:, -7:][:, np.arange(HORIZON) % 7]
            ti = x["decoder_time_idx"].cpu().numpy()
            n, h, _ = pred.shape
            result = pd.DataFrame({
                "store_nbr": np.repeat(idx.store_nbr.astype(str).to_numpy(), h),
                "item_nbr": np.repeat(idx.item_nbr.astype(str).to_numpy(), h),
                "origin_time_idx": np.repeat(ti[:,0]-1, h),
                "time_idx": ti.reshape(-1), "horizon": np.tile(np.arange(1,h+1),n),
                "seasonal_naive": baseline.reshape(-1),
            })
            if with_targets:
                result["actual"] = y[0].float().cpu().numpy().reshape(-1)
            flat = pred.reshape(-1, len(QUANTILES))
            clean = np.sort(np.maximum(flat, 0), axis=1)
            for j, col in enumerate(QCOLS):
                result["raw_" + col], result[col] = flat[:,j], clean[:,j]
            result["crossed_raw"] = (np.diff(flat, axis=1)<0).any(axis=1)
            result["date"] = START_DATE + pd.to_timedelta(result.time_idx, unit="D")
            result["origin_date"] = START_DATE + pd.to_timedelta(result.origin_time_idx, unit="D")
            rows.append(result)
    model.cpu()
    return pd.concat(rows, ignore_index=True)

val_pred = batch_forecasts(best_model, validation)
test_pred = batch_forecasts(best_model, testing)
assert val_pred.time_idx.max() <= VAL_CUTOFF
assert test_pred.time_idx.min() > VAL_CUTOFF
assert not test_pred.duplicated([*GROUP,"date"]).any()

def evaluate_predictions(df, cols=QCOLS):
    actual = df.actual.to_numpy()
    quant = df[cols].to_numpy()
    median = quant[:, QUANTILES.index(.5)]
    error = actual[:,None] - quant
    qs = np.asarray(QUANTILES)[None,:]
    result = {"MAE": np.abs(actual-median).mean(),
        "RMSE": np.sqrt(np.square(actual-median).mean()),
        "WAPE": np.abs(actual-median).sum()/np.abs(actual).sum() if np.abs(actual).sum()>0 else np.nan,
        "mean_pinball": np.maximum(qs*error, (qs-1)*error).mean(),
        "interval_coverage": ((actual>=quant[:,0]) & (actual<=quant[:,-1])).mean(),
        "interval_width": (quant[:,-1]-quant[:,0]).mean(), "n": len(df)}
    for q, col, values in zip(QUANTILES, QCOLS, quant.T):
        result["coverage_"+col] = (actual<=values).mean()
    return result

# 주간 기준선 + 검증 기간만으로 잔차 분위수 보정
val_pred["naive_residual"] = val_pred.actual - val_pred.seasonal_naive
residual_quantiles = val_pred.groupby("horizon").naive_residual.quantile(QUANTILES).unstack()
baseline_test = test_pred.copy()
for q, col in zip(QUANTILES, QCOLS):
    baseline_test[col] = np.maximum(0, baseline_test.seasonal_naive + baseline_test.horizon.map(residual_quantiles[q]))
baseline_test[QMED] = np.maximum(0, baseline_test[QMED])
metric_rows = [
    {"model":"TFT weather / raw", **evaluate_predictions(test_pred, ["raw_"+c for c in QCOLS])},
    {"model":"TFT weather / postprocessed", **evaluate_predictions(test_pred)},
    {"model":"Seasonal naive + validation residual calibration", **evaluate_predictions(baseline_test)},
]
metrics = pd.DataFrame(metric_rows)
display(metrics.round(4))
print("분위수 교차 비율(후처리 전):", round(test_pred.crossed_raw.mean(),4))
print("점 예측용 전주 기준선 MAE:", float((test_pred.actual-test_pred.seasonal_naive).abs().mean()))
print("Q10–Q90 목표 포함률:", QUANTILES[-1]-QUANTILES[0])
per_horizon = pd.DataFrame([{ "horizon":h, **evaluate_predictions(p)} for h,p in test_pred.groupby("horizon")])
display(per_horizon.round(4))
test_pred.to_csv(OUT / "test_predictions.csv", index=False)
val_pred.to_csv(OUT / "validation_predictions.csv", index=False)
metrics.to_csv(OUT / "test_metrics.csv", index=False)
per_horizon.to_csv(OUT / "metrics_by_horizon.csv", index=False)


In [ ]:
# 10. 선택 사항: 동일 조건에서 날씨 입력을 뺀 TFT 학습
if RUN_NO_WEATHER_ABLATION:
    tr0, va0, te0 = build_datasets(False)
    model0, path0 = fit_tft(tr0, va0, "tft_no_weather")
    pred0 = batch_forecasts(model0, te0)
    assert pred0[[*GROUP,"time_idx"]].equals(test_pred[[*GROUP,"time_idx"]])
    metrics = pd.concat([metrics, pd.DataFrame([{"model":"TFT no weather", **evaluate_predictions(pred0)}])], ignore_index=True)
    metrics.to_csv(OUT / "test_metrics.csv", index=False)
    pred0.to_csv(OUT / "no_weather_test_predictions.csv", index=False)
    display(metrics.round(4))
    print("날씨 유무 성능 차이는 예측 기여도입니다. 날씨의 인과효과나 매출 증가의 증명은 아닙니다.")
else:
    print("생략됨. 날씨 기여도를 실험하려면 RUN_NO_WEATHER_ABLATION=True 후 재실행하세요.")


In [ ]:
# 11. 예측구간 그래프 — 가장 많이 판매된 선택 시계열 1개
pair = selected.iloc[0]
example = test_pred[(test_pred.store_nbr==pair.store_nbr)&(test_pred.item_nbr==pair.item_nbr)].sort_values("date")
fig, ax = plt.subplots(figsize=(13,4))
ax.plot(example.date, example.actual, "o-", ms=3, label="Observed units", color="#172554")
ax.plot(example.date, example[QMED], label="TFT Q50", color="#0d9488")
ax.plot(example.date, example.seasonal_naive, "--", label="Previous week", alpha=.65)
ax.fill_between(example.date, example[QCOLS[0]], example[QCOLS[-1]], color="#5eead4", alpha=.25, label="Q10–Q90")
ax.set(title=f"Held-out test | store {pair.store_nbr}, item {pair.item_nbr}", ylabel="Units/day")
ax.legend(ncol=4); ax.grid(alpha=.15); fig.tight_layout()
fig.savefig(OUT / "forecast_example.png", dpi=160)
plt.show()


## 미래 7일 예측 / 실제 예보 입력
아래는 **데이터 마지막 관측일 다음 7일**입니다. 옛 Walmart 데이터를 쓰면 날짜도 과거입니다.
현재 국내 매장에 적용하려면 현재까지 이어지는 해당 매장의 판매·날씨 데이터로 다시 학습해야 합니다.

`FUTURE_WEATHER_CSV`를 비우면 검증 때와 같은 지속성 가정을 사용합니다.
예보를 넣을 때는 아래 형식으로 관측소×미래 날짜마다 한 행을 준비합니다.

`station_nbr,date,issued_at,tavg,tmax,tmin,preciptotal,avgspeed`

- `date`: 예보 대상 날짜, `issued_at`: 해당 예보를 발행한 시각(ISO 형식, 시간대 포함 권장).
- 같은 관측소·날짜별 여러 발행 이력은 **예측 시점까지 공개된 것 중 최신**을 선택합니다.
- 기본 컷오프는 자료 마지막 날의 영업 종료 시점입니다. 아래 `FORECAST_AS_OF`를 실제 시점에 맞추세요.
- 원자료와 동일 단위를 사용합니다. 예보 수치는 결측 없이 7일 모두 필요합니다.
- 지속성 가정으로 학습한 모델에 실제 예보를 넣으면 입력 분포가 바뀝니다. 운영 전 발행 이력이 있는
  과거 예보로 학습·평가를 다시 해야 합니다. `provided` 예보 결과에 기존 테스트 성능을 그대로 붙이지 마세요.


In [ ]:
# 12. 템플릿과 미래 입력 생성
future_dates = pd.date_range(END_DATE + pd.Timedelta(days=1), periods=HORIZON)
FORECAST_AS_OF = (END_DATE + pd.Timedelta(hours=23, minutes=59, seconds=59)).tz_localize("UTC")
# 미국 자료의 지역 시각은 비공개이므로 연구용 UTC 컷오프입니다. 실매장은 지역 시간대를 명시하세요.
weather_template = pd.MultiIndex.from_product([stations, future_dates], names=["station_nbr","date"]).to_frame(index=False)
weather_template["issued_at"] = ""
for col in WEATHER_BASE:
    weather_template[col] = np.nan
weather_template.to_csv(OUT / "future_weather_template.csv", index=False)

future = series_map.merge(pd.DataFrame({"date":future_dates}), how="cross")
last_w = weather[weather.date==END_DATE][["station_nbr", *WEATHER_FEATURES]]
future = future.merge(last_w, on="station_nbr", how="left", validate="many_to_one")
FORECAST_WEATHER_MODE = "persistence"
if FUTURE_WEATHER_CSV:
    fc = pd.read_csv(FUTURE_WEATHER_CSV, dtype={"station_nbr":str})
    require_columns(fc, ["station_nbr","date","issued_at",*WEATHER_BASE], "예보 CSV")
    fc["date"] = pd.to_datetime(fc.date, errors="raise").dt.normalize()
    fc["issued_at"] = pd.to_datetime(fc.issued_at, errors="raise", utc=True)
    assert fc.issued_at.notna().all(), "예보 발행 시각이 필요합니다."
    fc = fc[fc.issued_at <= FORECAST_AS_OF].sort_values("issued_at").drop_duplicates(["station_nbr","date"],keep="last")
    check = weather_template[["station_nbr","date"]].merge(fc, on=["station_nbr","date"], how="left",validate="one_to_one")
    for col in WEATHER_BASE:
        check[col] = pd.to_numeric(check[col], errors="raise")
    assert np.isfinite(check[WEATHER_BASE].to_numpy()).all(), "시점 이전에 발행된 완전한 7일 예보가 필요합니다."
    assert (check.tmin<=check.tavg).all() and (check.tavg<=check.tmax).all()
    assert (check[["preciptotal","avgspeed"]]>=0).all().all()
    for col in WEATHER_MISSING:
        check[col] = 0.0
    check["precip_trace"] = 0.0
    future = future.drop(columns=WEATHER_FEATURES).merge(check[["station_nbr","date",*WEATHER_FEATURES]],on=["station_nbr","date"],validate="many_to_one")
    FORECAST_WEATHER_MODE = "provided"
    print("실제 예보 입력 사용. 지속성 학습 대비 분포 변화 검증이 필요합니다.")
future["units"] = 0.0        # 미래 타깃 자리표시자. 추론용이며 학습·평가에 사용하지 않음
future["sales_observed"] = 0.0
future = calendar_features(future)
history = panel[panel.time_idx > MAX_TIME-ENCODER_DAYS].copy()
predict_frame = pd.concat([history, future], ignore_index=True).sort_values([*GROUP,"time_idx"])
prediction_ds = TimeSeriesDataSet.from_dataset(training, predict_frame, predict=True, stop_randomization=True)
future_pred = batch_forecasts(best_model, prediction_ds, weather_mode=FORECAST_WEATHER_MODE, with_targets=False)
assert len(future_pred)==len(selected)*HORIZON
future_pred["weather_mode"] = FORECAST_WEATHER_MODE
future_pred.to_csv(OUT / "future_sales_forecast.csv", index=False)
print("예측 대상:", future_dates.min().date(), "~", future_dates.max().date())
display(future_pred[[*GROUP,"date",*QCOLS,"weather_mode"]].head(14))


## 판매수량 → 발주 후보: 재고 정책의 가정을 명시
발주 결정에는 수요 예측 외에 재고·입고 일정·예약/미납 수량·납기·포장 단위가 필요합니다.
아래는 **주기적 검토의 order-up-to 근사 정책**입니다.

$$\text{재고 포지션}=\text{사용 가능 재고}+\text{적시 입고 예정}-\text{미납·예약 수량}$$
$$\text{발주 후보}=\left\lceil\frac{\max(0,\text{목표 재고}-\text{재고 포지션})}{\text{포장 단위}}\right\rceil\times\text{포장 단위}$$

보호 기간은 **납기 + 발주 검토 주기**입니다. 선택 분위수의 일별 수량을 보호 기간에 걸쳐 더해 목표 재고의 **휴리스틱**으로 씁니다.
**일별 Q90의 합은 보호 기간 누적 수요의 Q90이 아닙니다.** 아래 결과는 90% 서비스 수준 또는 최적 발주를 보장하지 않습니다.
정식 정책에는 누적 수요 직접 예측 또는 날짜 간 의존성을 반영한 시나리오 모의실험과 재고 시뮬레이션이 필요합니다.
Q50도 평균이 아니므로 Q50 합계는 '기대 수요'로 표기하지 않습니다.

재고 CSV에는 `store_nbr,item_nbr,on_hand,on_order_due,backorders,lead_time_days,review_days,pack_size,service_quantile`를 넣으세요.
`on_order_due`에는 보호 기간 안에 도착하는 수량만 넣고, 지연 입고·입고 전 품절·유통기한·최소발주량은 별도 검토하세요.
`service_quantile`은 학습한 분위수 중 하나입니다. 품절비용/과잉비용이 확인된 단일 기간 뉴스벤더 문제에서는
$q=C_u/(C_u+C_o)$가 출발점이지만, 여기의 여러 날짜 재고 정책에 그대로 최적성을 부여하지 않습니다.


In [ ]:
# 13. 재고 템플릿 + 권장 발주량 후보
inventory_template = selected.copy()
for col in ["on_hand","on_order_due","backorders","lead_time_days","review_days","pack_size","service_quantile"]:
    inventory_template[col] = np.nan
inventory_template.to_csv(OUT / "inventory_template.csv", index=False)

if INVENTORY_CSV:
    inventory = pd.read_csv(INVENTORY_CSV, dtype={c:str for c in GROUP})
    inventory_source = "user_inventory"
else:
    inventory = selected.copy()
    # 모두 예시 설정. 실제 재고 정보가 아님
    inventory["on_hand"] = 10
    inventory["on_order_due"] = 0
    inventory["backorders"] = 0
    inventory["lead_time_days"] = 1
    inventory["review_days"] = 1
    inventory["pack_size"] = 1
    inventory["service_quantile"] = 0.90
    inventory_source = "DEMO_NOT_FOR_ORDERING"
    print("주의: 예시 재고(보유 10개 등)입니다. 실제 주문에 사용하지 마세요.")

def recommend_orders(forecast, inventory, source):
    required = [*GROUP,"on_hand","on_order_due","backorders","lead_time_days","review_days","pack_size","service_quantile"]
    require_columns(inventory, required, "재고 CSV")
    assert not inventory.duplicated(GROUP).any(), "매장×상품별 재고 중복"
    inv = inventory.copy()
    for c in GROUP:
        inv[c] = inv[c].astype(str)
    numeric = required[2:]
    inv[numeric] = inv[numeric].apply(pd.to_numeric, errors="raise")
    assert np.isfinite(inv[numeric].to_numpy()).all(), "재고 값 누락/무한대"
    assert (inv[numeric] >= 0).all().all()
    for c in ["lead_time_days","review_days","pack_size"]:
        assert np.allclose(inv[c], inv[c].round()), f"{c}는 정수"
    assert (inv.review_days>=1).all() and (inv.pack_size>=1).all()
    assert ((inv.lead_time_days+inv.review_days)<=HORIZON).all(), "납기+검토 주기는 예측 기간 이하여야 합니다. HORIZON을 늘려 재학습하세요."
    found = forecast[GROUP].drop_duplicates().merge(inv, on=GROUP, how="left", validate="one_to_one")
    assert found[numeric].notna().all().all(), "예측 상품의 재고 입력 누락"
    results=[]
    for row in found.itertuples(index=False):
        qidx = [i for i,q in enumerate(QUANTILES) if np.isclose(row.service_quantile,q)]
        assert len(qidx)==1, f"service_quantile은 {QUANTILES} 중 선택"
        protection = int(row.lead_time_days+row.review_days)
        f = forecast[(forecast.store_nbr==row.store_nbr)&(forecast.item_nbr==row.item_nbr)&(forecast.horizon<=protection)]
        assert len(f)==protection
        target = float(f[QCOLS[qidx[0]]].sum())
        position = row.on_hand+row.on_order_due-row.backorders
        pack = int(row.pack_size)
        qty = int(math.ceil(max(0,target-position)/pack)*pack)
        results.append({"store_nbr":row.store_nbr,"item_nbr":row.item_nbr,
            "protection_days":protection,"selected_daily_quantile":row.service_quantile,
            "target_stock_heuristic":target,"inventory_position":position,
            "pack_size":pack,"recommended_order_units":qty,
            "inventory_source":source,"policy":"sum_daily_quantiles_heuristic"})
    return pd.DataFrame(results)

orders = recommend_orders(future_pred, inventory, inventory_source)
orders.to_csv(OUT / "recommended_orders.csv", index=False)
display(orders.head(20))


## 저장, 재실행, 문제 해결
결과 폴더에는 최적/마지막 체크포인트, 학습 로그, 테스트 지표, 예측 CSV, 발주 CSV, 입력 템플릿과 환경 기록이 저장됩니다.
Drive를 연결했다면 Colab 연결 종료 후에도 남습니다. 인증 토큰·원본 Kaggle 데이터는 결과 묶음에 넣지 않습니다.

- **CUDA out of memory**: `BATCH_SIZE=64` 또는 32, `HIDDEN_SIZE=32`로 변경하고 세션을 재시작하세요.
- **호스트 RAM 부족**: `MAX_SERIES`를 50으로 줄이세요. A100 GPU RAM과 Colab 호스트 RAM은 별개입니다.
- **import 오류**: 새 런타임에서 설치 셀을 실행하고 반드시 세션을 다시 시작하세요. `lightning.pytorch`와 별도 namespace를 섞지 마세요.
- **worker 오류/멈춤**: `NUM_WORKERS=0`으로 재시도하세요.
- **낮은 성능**: 먼저 전주 기준선, 0판매 비율, 품절·휴무, 예보 품질과 계절 이동을 확인하세요. TFT가 항상 우수한 것은 아닙니다.
- **검증 손실과 pinball 숫자가 다름**: 라이브러리의 2배 관례, 정규화, 후처리 여부를 구분하세요.
- **데이터 분할/모델 설정을 바꿀 때**: 설정 셀 이후를 다시 실행하세요. 테스트 지표로 반복 튜닝하면 최종 테스트로서의 의미가 약해집니다.

현재 평가는 선택된 상품과 정답이 존재하는 주간 구간에 한정됩니다. 대회에서 가린 날씨 민감 기간이 제외되면
극한 날씨 구간의 성능을 대표하지 못할 수 있습니다. 별도로 확보한 연속 실매장 데이터와 다수 시점의 백테스트가 필요합니다.


In [ ]:
# 14. 재현용 메타데이터 / 데이터셋 전처리 상태 / ZIP
config_names = ["SMOKE_TEST","SEED","ENCODER_DAYS","HORIZON","VALID_DAYS","TEST_DAYS",
    "MAX_SERIES","MIN_NONZERO_DAYS","BATCH_SIZE","MAX_EPOCHS","HIDDEN_SIZE","NUM_WORKERS",
    "TRAIN_BATCH_LIMIT","QUANTILES","RUN_NO_WEATHER_ABLATION"]
config = {name: globals()[name] for name in config_names}
config.update({"gpu":gpu,"precision":PRECISION,"start_date":str(START_DATE),"end_date":str(END_DATE),
    "train_end":str(TRAIN_END),"validation_end":str(VALID_END),
    "weather_training_mode":"persistence","future_weather_mode":FORECAST_WEATHER_MODE,
    "selected_series":len(selected),"train_windows":len(training),"validation_windows":len(validation),
    "test_windows":len(testing),"best_checkpoint":str(best_checkpoint),
    "test_raw_crossing_rate":float(test_pred.crossed_raw.mean()),"inventory_source":inventory_source,
    "packages": {p:version(p) for p in ["torch","pytorch-forecasting","lightning","pytorch-lightning","torchmetrics","pandas","numpy","scikit-learn"]}})
(OUT / "run_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
with open(OUT / "dataset_parameters.pkl", "wb") as f:
    pickle.dump(training.get_parameters(), f)
weather_defaults.to_json(OUT / "weather_fill_defaults.json", force_ascii=False)
residual_quantiles.to_csv(OUT / "baseline_validation_residual_quantiles.csv")
config["packages"]
import subprocess
(OUT / "environment_freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"], text=True), encoding="utf-8")
archive_path = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("완료:", OUT)
print("결과 묶음:", archive_path)
print("발주 파일:", OUT / "recommended_orders.csv")


In [ ]:
# 15. 선택: 결과 ZIP을 PC로 받기 (True로 바꿔 실행)
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive_path)


### 나중에 최적 모델 불러오기
같은 버전의 환경에서 설정·전처리·`WeatherCollator`/`make_loader` 정의를 준비하고 아래 형태로 로드합니다.
`pickle`과 체크포인트는 **본인이 만든 신뢰할 수 있는 파일만** 여세요.
새 데이터는 동일 스키마·단위로 정리하고 학습 때의 날짜 기준, 결측 대체값, encoder를 유지해야 합니다.

```python
with open(saved_run / "dataset_parameters.pkl", "rb") as f:
    parameters = pickle.load(f)
ds = TimeSeriesDataSet.from_parameters(parameters, prepared_frame, predict=True, stop_randomization=True)
model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_path, map_location="cpu")
# weather_mode는 persistence 또는 발행시각을 검증한 provided
pred = batch_forecasts(model, ds, weather_mode="persistence", with_targets=False)
```

새 매장/상품은 그룹 정규화와 embedding이 충분히 학습되지 않았으므로 신규 데이터로 재학습하는 편이 안전합니다.
운영에서는 데이터 수집 → 누락 검사 → 새 판매 이력 반영 → 예보 발행 시점 확인 → 예측 → 재고 정책 → 담당자 검토 순으로 연결하세요.

### 프로젝트 발표에서 주장할 수 있는 범위
1. 판매수량 예측 오차와 분위수 보정이 기준선보다 개선되는지 제시합니다.
2. 날씨 없는 TFT와 비교해 날씨 입력의 예측 기여도를 제시합니다.
3. 실제 발주 효과는 별도 재고 시뮬레이션/시범 운영으로 폐기율·품절률·매출·총비용을 측정합니다.
   현재 노트북의 예측 지표만으로 판매량 향상을 입증하지 않습니다.

### 참고 자료
- [Walmart Stormy Weather 데이터](https://www.kaggle.com/competitions/walmart-recruiting-sales-in-stormy-weather/data)
- [TFT 원 논문](https://arxiv.org/abs/1912.09363)
- [PyTorch Forecasting 1.5.0 TFT API](https://pytorch-forecasting.readthedocs.io/en/v1.5.0/api/pytorch_forecasting.models.temporal_fusion_transformer._tft.TemporalFusionTransformer.html)
- [TimeSeriesDataSet](https://pytorch-forecasting.readthedocs.io/en/v1.5.0/api/pytorch_forecasting.data.timeseries._timeseries.TimeSeriesDataSet.html)
- [QuantileLoss](https://pytorch-forecasting.readthedocs.io/en/v1.5.0/api/pytorch_forecasting.metrics.quantile.QuantileLoss.html)
- [Lightning 혼합 정밀도](https://lightning.ai/docs/pytorch/stable/common/precision_intermediate.html)
- [Kaggle 공식 CLI](https://github.com/Kaggle/kaggle-cli)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

**제작 시 확인 범위:** Python 3.12 / PyTorch 2.8.0 CPU에서 8개 매장×상품, 400일의 합성 데이터로
전처리 → TFT 학습 → 체크포인트 재로딩 → 검증·테스트 평가 → 미래 예측 → 발주 CSV 저장을 확인했습니다.
날씨 제외 비교 모델, 예보 발행 시점 제한, 사용자 재고 입력, 포장 단위 반올림, 미래 실제값 교란 시 예측 불변 검사와
노트북 형식 검증도 통과했습니다. 이 검사는 성능 벤치마크가 아닙니다.
원본 Kaggle 데이터 전체 학습과 A100 실행은 수행하지 않았으며, 그 성능 수치는 제공하지 않습니다.
